# 导出时间冲突接续与GNSS原始行

- 写入日期：2026-10-20
- 修改日期：2026-10-20
- 内容：按用户要求，在第二步结果目录生成一个便于逐条核对的CSV，不修改原始GNSS、第一步接续或链路结果。

## 1 说明

输入为正式接续删除记录、发生负时间间隔的年度第一步候选接续和同年度只读GNSS。范围为2021—2023年各年5月1日至6月25日，不限81城。负间隔指终点进入时间早于起点离开时间；等于0不属于本文件。

一行代表“一条冲突接续的一个端点访问包含的一条GNSS作业段”。原始行可能对应多条接续，会按对应关系重复列出，不代表原始数据重复。保留全部原接续字段和全部原GNSS字段，后者统一加GNSS_前缀；另外标明实际提供起点最晚结束或终点最早开始时间的原行。只作查看，不作为链路统计输入。

## 2 参数

依赖pandas、duckdb。项目根目录从当前目录向上寻找，无额外本地路径。默认拒绝覆盖已有查看文件；明确重跑时设置TIME_CONFLICT_ALLOW_OVERWRITE=1。普通重跑不自动归档。

In [1]:
from pathlib import Path
import hashlib
import os
import pandas as pd
import duckdb
from IPython.display import display

def find_project_root(start_path):
    for candidate in [Path(start_path).resolve(), *Path(start_path).resolve().parents]:
        if (candidate / 'README.md').exists() and (candidate / 'Data').is_dir():
            return candidate
    raise FileNotFoundError('没有找到项目根目录。')

project_root = find_project_root(Path.cwd())
output_path = project_root / 'Results/01_链路数据检查/时间冲突原始记录.csv'
manifest_path = project_root / 'Data/data_manifest.csv'
notebook_relative = 'Code/02_构建链路数据/export_time_conflict_rows.ipynb'
allow_overwrite = os.getenv('TIME_CONFLICT_ALLOW_OVERWRITE', '0') == '1'
if output_path.exists() and not allow_overwrite:
    raise FileExistsError('查看文件已存在；确认重跑时设置TIME_CONFLICT_ALLOW_OVERWRITE=1。')
source_paths = []
source_hashes = {}
all_edges, all_rows, all_visits, all_bounds = [], [], [], []
connection = duckdb.connect()
connection.execute('SET preserve_insertion_order=true')
sort_sql = 'date_value, start_ts, end_ts, center_ts NULLS LAST, info_id NULLS LAST, id NULLS LAST, source_row_no'

def file_fingerprint(path):
    with open(path, 'rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()

## 3 读取

先核对清单中的正式状态和文件指纹，再从删除记录选择负间隔接续。原始行号先对完整GNSS文件生成，再筛选农机，不能把筛选后的行号当作原文件行号。

In [2]:
manifest_table = pd.read_csv(manifest_path, dtype='string', encoding='utf-8-sig')
assert not manifest_table['relative_path'].duplicated().any()
manifest = manifest_table.set_index('relative_path')

def verify_source(relative_path):
    row = manifest.loc[relative_path]
    assert row['formal_status'] == '正式', relative_path
    path = project_root / relative_path
    digest = file_fingerprint(path)
    assert digest == row['sha256'], f'输入文件指纹不符：{relative_path}'
    assert path.stat().st_size == int(row['file_size_bytes']), f'输入文件大小不符：{relative_path}'
    source_paths.append(relative_path)
    source_hashes[relative_path] = digest
    return path

deletion_relative = 'Results/01_链路数据检查/接续删除记录.csv'
deletion_path = verify_source(deletion_relative)
deletion_input = pd.read_csv(deletion_path, encoding='utf-8-sig', dtype='string', keep_default_na=False)
assert len(deletion_input) == int(manifest.loc[deletion_relative, 'row_count'])
negative_edges = deletion_input.loc[pd.to_numeric(deletion_input['时间间隔小时'], errors='raise').lt(0)].copy()
assert not negative_edges['边记录编号'].duplicated().any()
assert len(negative_edges) > 0, '删除记录中没有负时间间隔的接续。'
conflict_years = sorted(negative_edges['年份'].astype(int).unique())
assert set(conflict_years).issubset({2021, 2022, 2023})
display(negative_edges.groupby('年份').size().rename('负间隔接续数').to_frame())

,负间隔接续数
年份,
2022,7
2023,25


## 4 处理

沿用第一步的完整排序及连续同城合并方法，找出冲突两端访问的全部GNSS原始行。首行号、末行号只作核对，不把两者之间的连续原文件行号范围当作访问成员。

In [3]:
for year in conflict_years:
    edge_relative = f'Data/03_有向边数据/{year}年农机城市接续.csv'
    edge_path = verify_source(edge_relative)
    edges = pd.read_csv(edge_path, encoding='utf-8-sig', dtype='string', keep_default_na=False)
    edges = edges.loc[pd.to_numeric(edges['时间间隔小时']) < 0].copy()
    raw_relative = f'Data/00_原始数据_只读/final_filtered_{str(year)[-2:]}_modified.csv'
    raw_path = verify_source(raw_relative)
    connection.register('wanted', pd.DataFrame({'did': edges['农机编号'].unique()}))
    # 原文件数据行从1开始编号，再选择农机，保持第一步的行号含义。
    raw = connection.execute("""
        SELECT * FROM (
          SELECT row_number() OVER () AS source_row_no, *
          FROM read_csv(?, header=true, all_varchar=true, parallel=false, ignore_errors=false))
        WHERE trim(did) IN (SELECT did FROM wanted)
        """, [str(raw_path)]).df()
    connection.register('selected_raw', raw)
    connection.execute("""
        CREATE OR REPLACE TEMP VIEW parsed AS
        SELECT source_row_no, NULLIF(trim(info_id),'') AS info_id, NULLIF(trim(id),'') AS id,
          trim(did) AS did, CAST(trim(date) AS DATE) AS date_value,
          CAST(trim(start_time) AS TIMESTAMP) AS start_ts, CAST(trim(end_time) AS TIMESTAMP) AS end_ts,
          COALESCE(try_strptime(trim(center_time),'%Y%m%d%H%M%S'),try_cast(trim(center_time) AS TIMESTAMP)) AS center_ts,
          NULLIF(trim(center_location),'') AS city FROM selected_raw
        """)
    connection.execute(f"""
        CREATE OR REPLACE TEMP VIEW flagged AS SELECT *,
          CASE WHEN lag(city) OVER(PARTITION BY did ORDER BY {sort_sql}) IS DISTINCT FROM city THEN 1 ELSE 0 END AS new_visit
        FROM parsed WHERE city IS NOT NULL
        """)
    visit_rows = connection.execute(f"""
        SELECT *,sum(new_visit) OVER(PARTITION BY did ORDER BY {sort_sql} ROWS UNBOUNDED PRECEDING)::BIGINT AS visit_order
        FROM flagged
        """).df()
    connection.register('visit_rows', visit_rows)
    visits = connection.execute(f"""
        SELECT did,visit_order,first(city ORDER BY {sort_sql}) AS city,min(start_ts) AS entry_time,
          max(end_ts) AS exit_time,count(*) AS record_count,
          first(source_row_no ORDER BY {sort_sql}) AS first_row,last(source_row_no ORDER BY {sort_sql}) AS last_row
        FROM visit_rows GROUP BY 1,2
        """).df().set_index(['did','visit_order'])
    affected_keys = set()
    for _, edge in edges.iterrows():
        for prefix in ['起点', '终点']:
            order = int(edge[prefix + '访问序号'])
            key = (edge['农机编号'], order)
            affected_keys.add(key)
            visit = visits.loc[key]
            # 不能仅依赖首末行范围；对照第一步完整访问定义逐项核对。
            assert str(visit.city) == edge[prefix + '城市']
            assert str(visit.entry_time) == edge[prefix + '进入时间']
            assert str(visit.exit_time) == edge[prefix + '离开时间']
            assert int(visit.record_count) == int(edge[prefix + '原始记录数'])
            assert int(visit.first_row) == int(edge[prefix + '首行号'])
            assert int(visit.last_row) == int(edge[prefix + '末行号'])
            members = visit_rows.loc[visit_rows.did.eq(key[0]) & visit_rows.visit_order.eq(order)]
            selected_boundary = (members.end_ts.eq(visit.exit_time) if prefix == '起点'
                                 else members.start_ts.eq(visit.entry_time))
            for _, boundary in members.loc[selected_boundary].iterrows():
                all_bounds.append({
                    '年份': year, '边记录编号': edge['边记录编号'], '农机编号': key[0],
                    '端点': prefix, '访问序号': order, '城市': visit.city,
                    '源数据行号': int(boundary.source_row_no),
                    'CSV行号': int(boundary.source_row_no) + 1,
                    '判断依据': '起点最晚结束' if prefix == '起点' else '终点最早开始',
                    '时间': str(boundary.end_ts if prefix == '起点' else boundary.start_ts)
                })
    members = visit_rows.loc[[key in affected_keys for key in zip(visit_rows.did,visit_rows.visit_order)]]
    selected_raw = raw.merge(members[['source_row_no','visit_order']],on='source_row_no',validate='one_to_one')
    selected_raw = selected_raw.rename(columns={'source_row_no':'源数据行号','visit_order':'访问序号'})
    selected_raw.insert(0,'年份',year)
    selected_raw.insert(1,'来源文件',raw_relative)
    selected_raw.insert(3,'CSV行号',selected_raw['源数据行号'] + 1)
    all_rows.extend(selected_raw.sort_values(['did','访问序号','start_time','end_time','源数据行号']).fillna('').to_dict('records'))
    for key in sorted(affected_keys):
        visit = visits.loc[key]
        all_visits.append({
            '年份': year, '农机编号': key[0], '访问序号': int(key[1]), '城市': visit.city,
            '进入时间': str(visit.entry_time), '离开时间': str(visit.exit_time),
            '原始记录数': int(visit.record_count), '首行号': int(visit.first_row), '末行号': int(visit.last_row)
        })
    all_edges.extend(edges.to_dict('records'))
connection.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 5 检查

逐端点核对城市、进入离开时间、原始记录数及首末行号。再核对每条边是否来自正式删除记录，以及实际提供判断时间的原始行。

In [4]:
edge_table = pd.DataFrame(all_edges)
raw_table = pd.DataFrame(all_rows)
visit_table = pd.DataFrame(all_visits)
boundary_table = pd.DataFrame(all_bounds)
assert set(edge_table['边记录编号']) == set(negative_edges['边记录编号'])
assert len(raw_table) == int(visit_table['原始记录数'].sum())
assert len(boundary_table) >= 2 * len(edge_table)
assert not raw_table[['年份','源数据行号']].duplicated().any()
assert not edge_table['边记录编号'].duplicated().any()
assert set(zip(boundary_table['年份'], boundary_table['源数据行号'])).issubset(set(zip(raw_table['年份'], raw_table['源数据行号'])))
boundary_lookup = boundary_table.merge(
    raw_table[['年份','源数据行号','did','center_location','start_time','end_time']],
    on=['年份','源数据行号'], how='left',validate='many_to_one')
assert boundary_lookup.did.eq(boundary_lookup['农机编号']).all()
assert boundary_lookup.center_location.str.strip().eq(boundary_lookup['城市']).all()
assert boundary_lookup.loc[boundary_lookup['端点'].eq('起点'),'end_time'].eq(boundary_lookup.loc[boundary_lookup['端点'].eq('起点'),'时间']).all()
assert boundary_lookup.loc[boundary_lookup['端点'].eq('终点'),'start_time'].eq(boundary_lookup.loc[boundary_lookup['端点'].eq('终点'),'时间']).all()
display(edge_table[['边记录编号','起点城市','终点城市','起点离开时间','终点进入时间','时间间隔小时']].head(7))
display(boundary_lookup[['年份','源数据行号','did','center_location','start_time','end_time']].head(4))
print(f'核对通过：{len(edge_table)}条冲突接续，{len(visit_table)}次访问，{len(raw_table)}条不同原始记录。')

,边记录编号,起点城市,终点城市,起点离开时间,终点进入时间,时间间隔小时
0,2022-642107101997-1-2-3,广元市,南充市,2022-05-17 18:48:11,2022-05-17 14:40:39,-4.125555555555556
1,2022-642107101997-1-2-4,广元市,广元市,2022-05-17 18:48:11,2022-05-17 17:50:05,-0.9683333333333334
2,2022-LOV0121112500122-1-1-2,驻马店市,信阳市,2022-05-25 16:49:17,2022-05-25 14:40:31,-2.1461111111111113
3,2022-LOV0122021101895-1-4-5,驻马店市,信阳市,2022-05-26 18:10:44,2022-05-26 16:09:12,-2.0255555555555556
4,2022-LOV0122030606653-1-1-2,枣庄市,临沂市,2022-05-23 19:19:19,2022-05-23 17:39:34,-1.6625
5,2022-LOV0122030606653-1-1-3,枣庄市,枣庄市,2022-05-23 19:19:19,2022-05-23 17:58:33,-1.346111111111111
6,2022-LOV0122042601971-1-5-6,信阳市,驻马店市,2022-05-26 07:59:07,2022-05-26 07:33:34,-0.42583333333333334


,年份,源数据行号,did,center_location,start_time,end_time
0,2022,2471755,642107101997,广元市,2022-05-17 12:12:52,2022-05-17 18:48:11
1,2022,2387748,642107101997,南充市,2022-05-17 14:40:39,2022-05-17 17:39:05
2,2022,2471755,642107101997,广元市,2022-05-17 12:12:52,2022-05-17 18:48:11
3,2022,2387749,642107101997,广元市,2022-05-17 17:50:05,2022-05-17 18:49:56


核对通过：32条冲突接续，54次访问，329条不同原始记录。


In [5]:
raw_metadata = {'年份', '来源文件', '源数据行号', 'CSV行号', '访问序号'}
raw_original_columns = [column for column in raw_table.columns if column not in raw_metadata]
boundary_keys = {(row['边记录编号'], row['端点'], int(row['源数据行号'])) for row in all_bounds}
export_rows = []
for edge in all_edges:
    year = int(edge['年份'])
    for endpoint_role in ['起点', '终点']:
        visit_order = int(edge[endpoint_role + '访问序号'])
        members = raw_table.loc[
            raw_table['年份'].eq(year) & raw_table['did'].eq(edge['农机编号']) &
            raw_table['访问序号'].eq(visit_order)]
        assert len(members) == int(edge[endpoint_role + '原始记录数'])
        for _, raw_record in members.iterrows():
            gnss_source_row_no = int(raw_record['源数据行号'])
            is_time_basis = (edge['边记录编号'], endpoint_role, gnss_source_row_no) in boundary_keys
            export_row = dict(edge)
            export_row.update({
                '接续来源文件': f'Data/03_有向边数据/{year}年农机城市接续.csv',
                'GNSS所属端点': endpoint_role, 'GNSS访问序号': visit_order,
                'GNSS源数据行号': gnss_source_row_no, 'GNSS_CSV行号': gnss_source_row_no + 1,
                'GNSS来源文件': raw_record['来源文件'], '是否提供判断时间': is_time_basis,
                '判断时间用途': ('起点最晚结束' if endpoint_role == '起点' else '终点最早开始')
                                if is_time_basis else '该访问的其他作业段',
            })
            export_row.update({f'GNSS_{column}': raw_record[column] for column in raw_original_columns})
            export_rows.append(export_row)
export_table = pd.DataFrame(export_rows)
front_columns = [
    '年份', '农机编号', '边记录编号', '起点城市', '终点城市', '起点访问序号', '终点访问序号',
    '边级', '衰减权重', '起点离开时间', '终点进入时间', '时间间隔小时',
    'GNSS所属端点', 'GNSS访问序号', 'GNSS_center_location', 'GNSS_start_time', 'GNSS_end_time',
    'GNSS源数据行号', 'GNSS_CSV行号', '是否提供判断时间', '判断时间用途', '接续来源文件', 'GNSS来源文件',
]
export_table = export_table[front_columns + [column for column in export_table.columns if column not in front_columns]]
assert not export_table[['边记录编号', 'GNSS所属端点', 'GNSS源数据行号']].duplicated().any()
assert export_table['边记录编号'].nunique() == len(edge_table)
assert len(export_table[['GNSS来源文件', 'GNSS源数据行号']].drop_duplicates()) == len(raw_table)
assert int(export_table['是否提供判断时间'].sum()) == len(boundary_table)
assert pd.to_numeric(export_table['时间间隔小时']).lt(0).all()
assert export_table['GNSS_did'].eq(export_table['农机编号']).all()
assert export_table['GNSS_CSV行号'].eq(export_table['GNSS源数据行号'] + 1).all()

## 6 输出

只生成一个UTF-8-SIG CSV：Results/01_链路数据检查/时间冲突原始记录.csv。一条原始GNSS行若关联多条冲突接续，会重复列出；按GNSS来源文件和GNSS源数据行号去重后才是不同原始行数。输出后读回逐格核对，并登记数据清单。

In [6]:
output_path.parent.mkdir(parents=True, exist_ok=True)
export_table.to_csv(output_path, index=False, encoding='utf-8-sig')
readback = pd.read_csv(output_path, encoding='utf-8-sig', dtype='string', keep_default_na=False)
pd.testing.assert_frame_equal(readback, export_table.astype('string'), check_dtype=False)
with open(output_path, 'rb') as handle:
    assert handle.read(3) == b'\xef\xbb\xbf'
for relative_path, digest in source_hashes.items():
    assert file_fingerprint(project_root / relative_path) == digest, f'输入在运行期间发生变化：{relative_path}'

manifest_record = {
    'file_id': 'TIME_CONFLICT_RAW_ROWS', 'relative_path': output_path.relative_to(project_root).as_posix(),
    'file_name': output_path.name, 'file_role': 'quality_output', 'formal_status': '正式',
    'read_only': 'false', 'year': '2021-2023', 'time_range': '各年05-01至06-25',
    'data_grain': '一行一条时间冲突接续的一个端点与一条GNSS作业段的对应关系',
    'row_count': str(len(export_table)), 'file_size_bytes': str(output_path.stat().st_size),
    'sha256': file_fingerprint(output_path), 'encoding': 'UTF-8-SIG',
    'source_or_generator': notebook_relative, 'upstream': '；'.join(source_paths),
    'purpose': '人工查看时间冲突接续及对应GNSS原始行；不作为链路汇总输入',
    'notes': f'{len(edge_table)}条冲突接续；{len(raw_table)}条不同GNSS行；共享原行按对应关系重复列出；2021年无负间隔接续',
}
updated_manifest = pd.concat([
    manifest_table.loc[manifest_table['file_id'].ne(manifest_record['file_id'])],
    pd.DataFrame([manifest_record], columns=manifest_table.columns)], ignore_index=True)
assert not updated_manifest['file_id'].duplicated().any()
assert not updated_manifest['relative_path'].duplicated().any()
updated_manifest.to_csv(manifest_path, index=False, encoding='utf-8-sig')
manifest_readback = pd.read_csv(manifest_path, encoding='utf-8-sig', dtype='string', keep_default_na=False)
pd.testing.assert_frame_equal(manifest_readback, updated_manifest.fillna('').astype('string'), check_dtype=False)
display(export_table.groupby('年份').agg(对应关系行数=('边记录编号','size'),冲突接续数=('边记录编号','nunique')))
print(f'输出{len(export_table)}行、{len(export_table.columns)}列；覆盖{len(edge_table)}条接续和{len(raw_table)}条不同GNSS记录。')
print(output_path)

,对应关系行数,冲突接续数
年份,,
2022,126,7
2023,232,25


输出358行、64列；覆盖32条接续和329条不同GNSS记录。
C:\Users\Administrator\Desktop\W\Results\01_链路数据检查\时间冲突原始记录.csv
